# 1. 주문관리(Order Management)

In [1]:
import sys, os, warnings

# 프로젝트 루트를 import 경로에 추가 (core, data 패키지 접근)
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

warnings.filterwarnings('ignore')
print(f'✅ 프로젝트 루트: {PROJECT_ROOT}')

✅ 프로젝트 루트: c:\dev\course\QuantPilot


In [2]:
from pathlib import Path
from core.trade.kis_broker import KisBroker
from core.trade.kis_constants import (
    CcldDvsn, OrdDvsn, SllBuyDvsnCd, TaxRate,
)

# core/trade/.env 에서 API 키/계좌 정보를 읽어 클라이언트 생성
client = KisBroker.from_env(env_path=Path(PROJECT_ROOT) / "core" / "trade" / ".env")
config = client.config

print("env      :", config.env)
print("base_url :", config.base_url)
print("계좌번호  :", config.domestic_stock.account_no)

env      : KisEnv.PAPER
base_url : BaseUrl.PAPER
계좌번호  : 50192156


### 매수/매도 주문 제출 

In [3]:
stock_code = "005930"   # 종목코드 (6자리)


# ══════════════════════════════════════════════════════════════
# 시장가: 매수 주문 → 체결 확인 → 매도 주문
# ══════════════════════════════════════════════════════════════
RUN_MARKET = True
if RUN_MARKET:
    print("── 1단계: 시장가 매수 ────────────────────────")
    buy = client.orders.buy_market(
        stock_code=stock_code,  # 종목코드
        quantity=1,             # 주문수량 (주)
    )
    odno = buy["output"]["ODNO"]   # 체결 확인에 사용할 주문번호
    print(f"  주문번호: {odno}")

    print("── 2단계: 체결 대기 (최대 20초) ─────────────")
    if client.orders.wait_for_fill(odno, timeout_sec=20):
        print("  ✓ 완전체결 확인!")

        # 매수 체결 후 즉시 시장가 매도
        print("── 3단계: 시장가 매도 ────────────────────────")
        sell = client.orders.sell_market(
            stock_code=stock_code,  # 종목코드
            quantity=1,             # 주문수량 (주)
        )
        print(f"  매도 주문번호: {sell['output']['ODNO']}")
    else:
        print("  ✗ 타임아웃 — 잔고 확인 후 수동 매도하세요.")


# ══════════════════════════════════════════════════════════════
# 지정가: 매수 주문 → 체결 확인 → 매도 주문
# ══════════════════════════════════════════════════════════════
RUN_LIMIT = True
if RUN_LIMIT:
    # 현재가 조회 후 지정가 설정
    price_data    = client.market.price(stock_code)
    current_price = int(price_data["output"]["stck_prpr"])
    buy_price     = current_price        # 현재가 지정가 → 빠른 체결 기대
    sell_price    = current_price + 500  # 목표 매도가

    print(f"── 1단계: 지정가 매수 {buy_price:,}원 ───────────")
    buy = client.orders.buy_limit(
        stock_code=stock_code,  # 종목코드
        quantity=1,             # 주문수량 (주)
        price=buy_price,        # 주문단가 (원) — 지정가이므로 0 불가
    )
    odno = buy["output"]["ODNO"]
    print(f"  주문번호: {odno}")

    print("── 2단계: 체결 대기 (최대 30초) ─────────────")
    if client.orders.wait_for_fill(odno, timeout_sec=30):
        print("  ✓ 완전체결 확인!")

        # 매수 체결 확인 후 지정가 매도 제출
        print(f"── 3단계: 지정가 매도 {sell_price:,}원 ───────────")
        sell = client.orders.sell_limit(
            stock_code=stock_code,  # 종목코드
            quantity=1,             # 주문수량 (주)
            price=sell_price,       # 주문단가 (원)
        )
        print(f"  매도 주문번호: {sell['output']['ODNO']}")
    else:
        print("  ✗ 타임아웃 — 주문 취소 또는 수동 매도하세요.")


if not RUN_MARKET and not RUN_LIMIT:
    print("RUN_MARKET 또는 RUN_LIMIT = True 로 바꾸면 실행됩니다.")
    print("  RUN_MARKET : 시장가 매수 → 체결 확인 → 시장가 매도")
    print("  RUN_LIMIT  : 지정가 매수 → 체결 확인 → 지정가 매도")

── 1단계: 시장가 매수 ────────────────────────
  주문번호: 0000017692
── 2단계: 체결 대기 (최대 20초) ─────────────
  [ 2s] 체결:1주  잔량:0주
  ✓ 완전체결 확인!
── 3단계: 시장가 매도 ────────────────────────
  매도 주문번호: 0000017702
── 1단계: 지정가 매수 338,000원 ───────────
  주문번호: 0000017706
── 2단계: 체결 대기 (최대 30초) ─────────────
  [ 2s] 체결:1주  잔량:0주
  ✓ 완전체결 확인!
── 3단계: 지정가 매도 338,500원 ───────────
  매도 주문번호: 0000017718


### 주문 취소/수정 

In [4]:
import time

stock_code = "005930"   # 종목코드 (6자리)


# ══════════════════════════════════════════════════════════════
# 취소 테스트: 지정가 주문 제출 → 미체결 확인 → 전량 취소
# ══════════════════════════════════════════════════════════════
RUN_CANCEL_TEST = True
if RUN_CANCEL_TEST:
    # 현재가보다 훨씬 낮은 가격으로 주문 → 즉시 체결되지 않음
    price_data    = client.market.price(stock_code)
    current_price = int(price_data["output"]["stck_prpr"])
    test_price    = current_price - 10_000  # 현재가보다 훨씬 낮게 → 체결 안 됨

    print(f"── 1단계: 지정가 매수 {test_price:,}원 (체결 안 되는 가격) ─")
    order = client.orders.buy_limit(
        stock_code=stock_code,  # 종목코드
        quantity=1,             # 주문수량 (주)
        price=test_price,       # 주문단가 (원)
    )
    # 취소/정정에 필요한 두 개의 주문 식별자 추출
    odno    = order["output"]["ODNO"]                # 주문번호 — 취소/정정 시 필수
    krx_org = order["output"]["KRX_FWDG_ORD_ORGNO"] # 지점번호 — 취소/정정 시 필수
    print(f"  주문번호: {odno}  지점: {krx_org}")

    time.sleep(0.5)  # 거래소에 주문이 등록될 때까지 잠시 대기

    print("── 2단계: 전량 취소 ─────────────────────────")
    result = client.orders.cancel(
        orgn_odno=odno,              # 원주문번호 — 취소 대상 주문번호
        krx_fwdg_ord_orgno=krx_org, # 한국거래소 전송 주문 조직번호 (지점번호)
        qty_all_yn="Y",             # 잔량전부주문여부: "Y"=전량취소, "N"=일부취소
    )
    print(f"  취소 완료: {result['output']}")


# ══════════════════════════════════════════════════════════════
# 정정 테스트: 지정가 주문 제출 → 가격 정정
# ══════════════════════════════════════════════════════════════
RUN_MODIFY_TEST = True
if RUN_MODIFY_TEST:
    # 체결 안 되는 가격으로 주문 후 가격을 정정
    price_data    = client.market.price(stock_code)
    current_price = int(price_data["output"]["stck_prpr"])
    test_price    = current_price - 10_000  # 체결 안 되는 가격
    new_price     = current_price - 9_000   # 정정할 가격 (원래보다 1,000원 높임)

    print(f"── 1단계: 지정가 매수 {test_price:,}원 (체결 안 되는 가격) ─")
    order = client.orders.buy_limit(
        stock_code=stock_code,  # 종목코드
        quantity=1,             # 주문수량 (주)
        price=test_price,       # 주문단가 (원)
    )
    odno    = order["output"]["ODNO"]
    krx_org = order["output"]["KRX_FWDG_ORD_ORGNO"]
    print(f"  주문번호: {odno}  지점: {krx_org}")

    time.sleep(0.5)  # 거래소에 주문이 등록될 때까지 잠시 대기

    print(f"── 2단계: 가격 정정 {test_price:,}원 → {new_price:,}원 ─────")
    result = client.orders.modify(
        orgn_odno=odno,              # 원주문번호 — 정정 대상 주문번호
        krx_fwdg_ord_orgno=krx_org, # 한국거래소 전송 주문 조직번호 (지점번호)
        ord_qty=1,                  # 정정 주문수량 (주)
        ord_unpr=new_price,         # 정정 주문단가 (원)
    )
    print(f"  정정 완료: {result['output']}")


if not RUN_CANCEL_TEST and not RUN_MODIFY_TEST:
    print("RUN_CANCEL_TEST 또는 RUN_MODIFY_TEST = True 로 바꾸면 실행됩니다.")
    print("  RUN_CANCEL_TEST : 지정가 주문 제출 → 즉시 전량 취소")
    print("  RUN_MODIFY_TEST : 지정가 주문 제출 → 가격 정정")

── 1단계: 지정가 매수 327,500원 (체결 안 되는 가격) ─
  주문번호: 0000017728  지점: 00950
── 2단계: 전량 취소 ─────────────────────────
  취소 완료: {'KRX_FWDG_ORD_ORGNO': '00950', 'ODNO': '0000017731', 'ORD_TMD': '104509', 'SOR_ODNO': ''}
── 1단계: 지정가 매수 327,500원 (체결 안 되는 가격) ─
  주문번호: 0000017744  지점: 00950
── 2단계: 가격 정정 327,500원 → 328,500원 ─────
  정정 완료: {'KRX_FWDG_ORD_ORGNO': '00950', 'ODNO': '0000017751', 'ORD_TMD': '104521', 'SOR_ODNO': ''}


### 주문 상태 조회(체결 대기, 부분 체결, 완전 체결)

In [5]:
from datetime import date

today = date.today().strftime("%Y%m%d")

# 오늘 하루의 전체 주문 내역 조회 (체결·미체결·취소 모두 포함)
orders = client.history.get(
    start_date=today,
    end_date=today,
    sll_buy_dvsn_cd=SllBuyDvsnCd.ALL,   # 전체 (매수+매도)
    ccld_dvsn=CcldDvsn.ALL,              # 전체 (체결+미체결)
)

output1 = orders.get("output1", [])
print(f"=== 오늘({today}) 주문 현황 ({len(output1)}건) ===")

for o in output1:
    tot_ccld = int(o.get("tot_ccld_qty", "0") or "0")  # 총체결수량
    ord_qty  = int(o.get("ord_qty",      "0") or "0")  # 주문수량
    rmn_qty  = int(o.get("rmn_qty",      "0") or "0")  # 잔여수량

    # 4가지 상태 분류: 취소 > 미체결 > 부분체결 > 완전체결
    if o.get("cncl_yn") == "Y":     # 취소여부: "Y"=취소됨
        status = "취소"
    elif tot_ccld == 0:
        status = "미체결"
    elif rmn_qty > 0:
        status = "부분체결"
    else:
        status = "완전체결"

    side = "매수" if o.get("sll_buy_dvsn_cd") == SllBuyDvsnCd.BUY else "매도"
    print(
        f"[{status}] [{side}] {o.get('prdt_name')}({o.get('pdno')}) | "
        f"주문:{ord_qty}주 체결:{tot_ccld}주 잔량:{rmn_qty}주 | "
        f"주문시각:{o.get('ord_tmd')}"
    )

=== 오늘(20260616) 주문 현황 (8건) ===
[미체결] [매수] 삼성전자(005930) | 주문:1주 체결:0주 잔량:1주 | 주문시각:104521
[미체결] [매수] 삼성전자(005930) | 주문:1주 체결:0주 잔량:0주 | 주문시각:104517
[취소] [매수] 삼성전자(005930) | 주문:1주 체결:0주 잔량:0주 | 주문시각:104509
[미체결] [매수] 삼성전자(005930) | 주문:1주 체결:0주 잔량:0주 | 주문시각:104505
[미체결] [매도] 삼성전자(005930) | 주문:1주 체결:0주 잔량:1주 | 주문시각:104459
[완전체결] [매수] 삼성전자(005930) | 주문:1주 체결:1주 잔량:0주 | 주문시각:104455
[완전체결] [매도] 삼성전자(005930) | 주문:1주 체결:1주 잔량:0주 | 주문시각:104450
[완전체결] [매수] 삼성전자(005930) | 주문:1주 체결:1주 잔량:0주 | 주문시각:104445


# 2. 포지션 & 잔고 조회(Account Info)

### 현재 보유 종목 및 수량 

In [6]:
# 계좌 잔고 전체 조회 (output1=보유종목, output2=계좌 요약)
balance  = client.account.balance()
holdings = balance.get("output1", [])  # 보유 종목 건별 목록 (array)

print(f"=== 보유 종목 ({len(holdings)}개) ===")
if not holdings:
    print("보유 종목 없음")

# 보유 종목 한 줄씩 출력
for h in holdings:
    hldg_qty  = int(h.get("hldg_qty",       "0") or "0")  # 보유수량
    ord_psbl  = int(h.get("ord_psbl_qty",   "0") or "0")  # 주문가능수량 (보유 중 매도 가능)
    pchs_avg  = float(h.get("pchs_avg_pric","0") or "0")  # 매입평균단가 (원)
    prpr      = int(h.get("prpr",           "0") or "0")  # 현재가 (원)
    evlu_pfls = int(h.get("evlu_pfls_amt",  "0") or "0")  # 평가손익금액 (원)
    pfls_rt   = h.get("evlu_pfls_rt", "0")                 # 평가손익율 (%)

    print(f"\n종목: {h.get('prdt_name')} ({h.get('pdno')})")
    print(f"  보유수량:   {hldg_qty:>8,}주  (주문가능: {ord_psbl:,}주)")
    print(f"  매입평균:   {pchs_avg:>10,.0f}원")
    print(f"  현재가:     {prpr:>10,}원")
    print(f"  평가손익:   {evlu_pfls:>+10,}원  ({pfls_rt}%)")

=== 보유 종목 (2개) ===

종목: 삼성전자 (005930)
  보유수량:          3주  (주문가능: 2주)
  매입평균:      323,859원
  현재가:        337,000원
  평가손익:      +39,421원  (4.06%)

종목: KODEX 단기채권 (153130)
  보유수량:      1,953주  (주문가능: 1,953주)
  매입평균:      113,302원
  현재가:        113,135원
  평가손익:     -325,349원  (-0.15%)


### 가용 현금 잔고 

In [7]:
# output2는 단일 dict를 리스트로 감싼 형태 → [0]으로 꺼냄
summary = balance.get("output2", [{}])[0]

# 예수금 3단계: D+0(당일) / D+1(익일) / D+2(모레) — 매수가능 기준은 D+2
dnca = int(summary.get("dnca_tot_amt",       "0") or "0")  # 예수금총금액
d1   = int(summary.get("nxdy_excc_amt",       "0") or "0")  # D+1 익일정산금액
d2   = int(summary.get("prvs_rcdl_excc_amt", "0") or "0")  # D+2 가수도정산금액

print("=== 가용 현금 잔고 ===")
print(f"예수금(D+0): {dnca:>15,}원")
print(f"D+1 정산금: {d1:>15,}원")
print(f"D+2 정산금: {d2:>15,}원")

# 특정 종목에 대한 매수가능 금액/수량을 별도 API로 정밀 조회
stock_code = "005930"
buyable = client.account.buyable_amount(
    stock_code=stock_code,
    ord_unpr="",               # 주문단가: 시장가 조회 시 공란, 지정가 조회 시 가격 입력
    ord_dvsn=OrdDvsn.MARKET,   # 주문구분: 시장가
)
out = buyable.get("output", {})
print(f"\n=== {stock_code} 매수가능 조회 ===")
print(f"주문가능현금:      {int(out.get('ord_psbl_cash',  '0') or '0'):>15,}원")
print(f"미수없는 매수금액: {int(out.get('nrcvb_buy_amt',  '0') or '0'):>15,}원")  # 미수 미사용 기준
print(f"미수없는 매수수량: {int(out.get('nrcvb_buy_qty',  '0') or '0'):>15,}주")  # 미수 미사용 기준

=== 가용 현금 잔고 ===
예수금(D+0):     278,532,243원
D+1 정산금:     278,532,243원
D+2 정산금:     278,193,187원

=== 005930 매수가능 조회 ===
주문가능현금:          276,802,221원
미수없는 매수금액:     277,825,197원
미수없는 매수수량:             631주


### 총 평가 금액 

In [8]:
# 앞 셀(가용 현금 잔고)에서 로드한 summary 재사용
summary = balance.get("output2", [{}])[0]  # 계좌 전체 요약 (단일 dict)

# 금액 필드를 정수로 변환 (API는 모든 수치를 문자열로 반환)
tot_evlu  = int(summary.get("tot_evlu_amt",           "0") or "0")  # 총평가금액 (예수금 + 유가증권 평가)
evlu_smtl = int(summary.get("evlu_amt_smtl_amt",      "0") or "0")  # 유가증권 평가금액 합계
pchs_smtl = int(summary.get("pchs_amt_smtl_amt",      "0") or "0")  # 매입금액 합계
pfls_smtl = int(summary.get("evlu_pfls_smtl_amt",     "0") or "0")  # 평가손익 합계금액
nass      = int(summary.get("nass_amt",                "0") or "0")  # 순자산금액 (총평가 - 대출)
bfdy_tot  = int(summary.get("bfdy_tot_asst_evlu_amt", "0") or "0")  # 전일 총자산 평가금액
asst_icdc = int(summary.get("asst_icdc_amt",          "0") or "0")  # 자산증감액 (당일 - 전일)

print("=== 총 평가 금액 ===")
print(f"총 평가금액:     {tot_evlu:>15,}원")
print(f"  유가증권 평가: {evlu_smtl:>15,}원")
print(f"  매입금액:      {pchs_smtl:>15,}원")
print(f"  평가손익:      {pfls_smtl:>+15,}원")
print(f"순자산금액:      {nass:>15,}원")
print(f"전일 총자산:     {bfdy_tot:>15,}원")
print(f"자산증감액:      {asst_icdc:>+15,}원")

=== 총 평가 금액 ===
총 평가금액:         500,156,842원
  유가증권 평가:     221,963,655원
  매입금액:          222,249,582원
  평가손익:             -285,927원
순자산금액:          500,156,842원
전일 총자산:         500,168,663원
자산증감액:              -11,821원


# 3. 시세 데이터 수신(Market Data)

### 현재가 조회 

In [9]:
STOCK_CODE = "005930"   # 종목코드 (6자리)
TICK_COUNT = 10         # 수신할 체결 틱 수 (이 수만큼 받으면 종료)

await client.market.stream_price(STOCK_CODE, count=TICK_COUNT)

[005930] 실시간 현재가 구독 시작 (H0STCNT0, 10틱)...

   시각            현재가    등락      전일대비       비율         체결량  구분
-----------------------------------------------------------------
10:45:45     337,500  ↑       +500    0.15%           3  매수
10:45:45     337,000  -         +0    0.00%          56  매도
10:45:45     337,000  -         +0    0.00%           5  매도
10:45:46     337,500  ↑       +500    0.15%          55  매수
10:45:46     337,500  ↑       +500    0.15%           1  매수
10:45:47     337,500  ↑       +500    0.15%           1  매수
10:45:47     337,500  ↑       +500    0.15%           3  매수
10:45:47     337,250  ↑       +250    0.07%           8  매수
10:45:48     337,250  ↑       +250    0.07%          42  매수
10:45:48     337,250  ↑       +250    0.07%           1  매수


### 호가창(bid/ask)

In [10]:
LEVELS       = 5    # 표시할 호가 단계 (1~10 사이 값)
UPDATE_COUNT = 5    # 수신할 호가 업데이트 횟수 (이 수만큼 받으면 종료)

await client.market.stream_orderbook(STOCK_CODE, count=UPDATE_COUNT, levels=LEVELS)

[005930] 실시간 호가창 구독 시작 (H0STASP0, 5회 업데이트)...

=== 호가창 [005930]  10:45:50 ===
          매도호가          잔량
  --------------------------
       340,000원      52,674주
       339,500원      38,685주
       339,000원      19,241주
       338,500원       8,620주
       338,000원      11,345주
       337,500원       5,356주
       337,000원      48,701주
       336,500원      17,874주
       336,000원      22,773주
       335,500원      20,817주
  --------------------------
  총매도잔량: 240,962주 / 총매수잔량: 268,990주

=== 호가창 [005930]  10:45:50 ===
          매도호가          잔량
  --------------------------
       340,000원      52,674주
       339,500원      38,685주
       339,000원      19,241주
       338,500원       8,620주
       338,000원      11,345주
       337,500원       5,578주
       337,000원      48,556주
       336,500원      17,874주
       336,000원      22,774주
       335,500원      20,817주
  --------------------------
  총매도잔량: 240,962주 / 총매수잔량: 269,068주

=== 호가창 [005930]  10:45:51 ===
          매도호가          잔량
  -------

### 실시간 체결 데이터(웹소캣)

In [11]:
MAX_EVENTS = 30  # 총 수신 이벤트 수 (체결 + 호가 합산, 이 수만큼 받으면 종료)

await client.market.stream_realtime(STOCK_CODE, max_events=MAX_EVENTS)

[005930] 실시간 체결(H0STCNT0) + 호가(H0STASP0) 동시 구독 시작...
─────────────────────────────────────────────────────────────────
[호가] 10:45:54  매도1:   338,000원(12,227주)  매수1:   337,500원(6,406주)  스프레드:500원
[체결] 10:45:54     337,500원  ↑  +500  체결량:       3주  매도
[호가] 10:45:54  매도1:   338,000원(12,227주)  매수1:   337,500원(6,406주)  스프레드:500원
[호가] 10:45:54  매도1:   338,000원(12,229주)  매수1:   337,500원(6,396주)  스프레드:500원
[체결] 10:45:54     337,500원  ↑  +500  체결량:       4주  매도
[호가] 10:45:54  매도1:   338,000원(12,229주)  매수1:   337,500원(6,384주)  스프레드:500원
[체결] 10:45:54     337,500원  ↑  +500  체결량:      10주  매도
[호가] 10:45:55  매도1:   338,000원(12,236주)  매수1:   337,500원(6,324주)  스프레드:500원
[호가] 10:45:55  매도1:   338,000원(12,236주)  매수1:   337,500원(6,308주)  스프레드:500원
[체결] 10:45:55     337,500원  ↑  +500  체결량:       3주  매도
[호가] 10:45:55  매도1:   338,000원(12,236주)  매수1:   337,500원(6,255주)  스프레드:500원
[체결] 10:45:55     337,500원  ↑  +500  체결량:      10주  매도
[호가] 10:45:55  매도1:   338,000원(12,242주)  매수1:   337,500원(6,177주)  스프레드:500

# 4. 체결 내역 조회(Execution History)

### 오늘 체결 내역 

In [12]:
from datetime import date

today = date.today().strftime("%Y%m%d")

# 당일 체결 건만 조회 (CcldDvsn.FILLED 로 미체결·취소 제외)
executions = client.history.get(
    start_date=today,
    end_date=today,
    sll_buy_dvsn_cd=SllBuyDvsnCd.ALL,   # 전체 (매수+매도)
    ccld_dvsn=CcldDvsn.FILLED,           # 체결만 (미체결 제외)
)

output1 = executions.get("output1", [])   # 체결 건별 내역 (array)
output2 = executions.get("output2", {})   # 합계 요약 (single)

print(f"=== 오늘({today}) 체결 내역 ({len(output1)}건) ===")

# 체결 건 별 상세 출력
for e in output1:
    side     = "매수" if e.get("sll_buy_dvsn_cd") == SllBuyDvsnCd.BUY else "매도"
    ccld_qty = int(e.get("tot_ccld_qty", "0") or "0")  # 총체결수량
    ccld_amt = int(e.get("tot_ccld_amt", "0") or "0")  # 총체결금액
    avg_prc  = int(e.get("avg_prvs",     "0") or "0")  # 평균단가
    print(
        f"[{side}] {e.get('prdt_name')}({e.get('pdno')}) | "
        f"체결수량:{ccld_qty:,}주 | 평균단가:{avg_prc:,}원 | "
        f"체결금액:{ccld_amt:,}원 | 시각:{e.get('ord_tmd')}"
    )

# 당일 합계 요약 출력
if output2:
    tot_qty = output2.get("tot_ccld_qty", "0")               # 당일 총체결수량
    tot_amt = int(output2.get("tot_ccld_amt", "0") or "0")   # 당일 총체결금액
    print(f"\n당일 합계 → 체결수량: {tot_qty}주 | 체결금액: {tot_amt:,}원")

=== 오늘(20260616) 체결 내역 (3건) ===
[매수] 삼성전자(005930) | 체결수량:1주 | 평균단가:338,000원 | 체결금액:338,000원 | 시각:104455
[매도] 삼성전자(005930) | 체결수량:1주 | 평균단가:338,000원 | 체결금액:338,000원 | 시각:104450
[매수] 삼성전자(005930) | 체결수량:1주 | 평균단가:338,250원 | 체결금액:338,250원 | 시각:104445

당일 합계 → 체결수량: 3주 | 체결금액: 1,014,250원


### 기간별 거래 내역

In [13]:
from datetime import date, timedelta

DAYS     = 30
end_dt   = date.today()
start_dt = end_dt - timedelta(days=DAYS)  # 오늘로부터 30일 전

# 기간 내 체결 건만 조회
history = client.history.get(
    start_date=start_dt.strftime("%Y%m%d"),
    end_date=end_dt.strftime("%Y%m%d"),
    sll_buy_dvsn_cd=SllBuyDvsnCd.ALL,   # 전체 (매수+매도)
    ccld_dvsn=CcldDvsn.FILLED,           # 체결만 (미체결 제외)
)

output1 = history.get("output1", [])   # 체결 건별 내역 (array)
output2 = history.get("output2", {})   # 기간 합계 요약 (single)

print(f"=== 기간별 체결 내역 ({start_dt} ~ {end_dt}, {len(output1)}건) ===")

# 체결 건 별 날짜·종목·수량·금액 출력
for e in output1:
    side    = "매수" if e.get("sll_buy_dvsn_cd") == SllBuyDvsnCd.BUY else "매도"
    qty     = int(e.get("tot_ccld_qty", "0") or "0")  # 총체결수량
    amt     = int(e.get("tot_ccld_amt", "0") or "0")  # 총체결금액
    avg_prc = int(e.get("avg_prvs",     "0") or "0")  # 평균단가
    print(
        f"{e.get('ord_dt')} [{side}] {e.get('prdt_name')} | "
        f"{qty:,}주 @ {avg_prc:,}원 | 합계:{amt:,}원"
    )

# 기간 합계 요약 출력
if output2:
    tot_qty = output2.get("tot_ccld_qty", "0")                  # 기간 총체결수량
    tot_amt = int(output2.get("tot_ccld_amt",    "0") or "0")   # 기간 총체결금액
    prsm    = int(output2.get("prsm_tlex_smtl",  "0") or "0")   # 추정제비용합계 (수수료+세금 추정)
    print(f"\n합계 → 총체결수량:{tot_qty}주 | 총체결금액:{tot_amt:,}원 | 추정제비용:{prsm:,}원")

=== 기간별 체결 내역 (2026-05-17 ~ 2026-06-16, 15건) ===
20260616 [매수] 삼성전자 | 1주 @ 338,000원 | 합계:338,000원
20260616 [매도] 삼성전자 | 1주 @ 338,000원 | 합계:338,000원
20260616 [매수] 삼성전자 | 1주 @ 338,250원 | 합계:338,250원
20260610 [매수] KODEX 단기채권 | 1,953주 @ 113,301원 | 합계:221,278,004원
20260609 [매도] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매수] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매도] 삼성전자 | 1주 @ 307,000원 | 합계:307,000원
20260609 [매수] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매도] 삼성전자 | 1주 @ 306,500원 | 합계:306,500원
20260609 [매수] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매도] 삼성전자 | 1주 @ 306,500원 | 합계:306,500원
20260609 [매수] 삼성전자 | 1주 @ 305,500원 | 합계:305,500원
20260609 [매도] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매수] 삼성전자 | 1주 @ 306,000원 | 합계:306,000원
20260609 [매도] 삼성전자 | 1주 @ 307,500원 | 합계:307,500원

합계 → 총체결수량:1992주 | 총체결금액:233,315,754원 | 추정제비용:45,514원


### 수수료 정보

In [14]:
from datetime import date, timedelta

# KIS API는 별도의 수수료 조회 엔드포인트를 제공하지 않습니다.
# 수수료는 체결 내역 summary의 prsm_tlex_smtl(추정제비용합계)로 추정합니다.

# ── 수수료 기본 정보 ──────────────────────────────────────────────────────
print("=== 국내주식 수수료 안내 ===")
print(f"  온라인 매매 수수료: {TaxRate.COMMISSION.rate*100:.3f}%")
print(f"  증권거래세 (매도시): {TaxRate.SECURITIES_TAX.rate*100:.2f}%")
print()

# ── 최근 30일 추정 제비용 조회 ────────────────────────────────────────────
end_dt   = date.today()
start_dt = end_dt - timedelta(days=30)

result = client.history.get(
    start_date=start_dt.strftime("%Y%m%d"),
    end_date=end_dt.strftime("%Y%m%d"),
    ccld_dvsn=CcldDvsn.FILLED,   # 체결만
)
output2 = result.get("output2", {})   # 기간 합계 요약

if output2:
    tot_amt = int(output2.get("tot_ccld_amt",   "0") or "0")  # 기간 총체결금액
    prsm    = int(output2.get("prsm_tlex_smtl", "0") or "0")  # 추정제비용합계 (수수료+세금 추정치)
    print(f"최근 30일 총 체결금액:  {tot_amt:>15,}원")
    print(f"최근 30일 추정 제비용:  {prsm:>15,}원")
    if tot_amt > 0:
        effective = prsm / tot_amt * 100
        print(f"실효 비용률:            {effective:>15.4f}%")
else:
    print("체결 내역 없음 → 수수료 데이터 없음")

# ── 수수료 간이 계산기 ────────────────────────────────────────────────────
trade_amount = 1_000_000  # 100만원 거래 예시
commission   = trade_amount * TaxRate.COMMISSION.rate      # 매수·매도 각각 발생
tax          = trade_amount * TaxRate.SECURITIES_TAX.rate  # 매도 시에만 발생
print(f"\n[계산 예시] 거래금액 {trade_amount:,}원 기준")
print(f"  예상 수수료(매수): {commission:>8,.0f}원")
print(f"  예상 수수료(매도): {commission:>8,.0f}원")
print(f"  증권거래세(매도):  {tax:>8,.0f}원")
print(f"  총 비용(매도):     {commission + tax:>8,.0f}원")

=== 국내주식 수수료 안내 ===
  온라인 매매 수수료: 0.015%
  증권거래세 (매도시): 0.20%

최근 30일 총 체결금액:      233,315,754원
최근 30일 추정 제비용:           45,514원
실효 비용률:                     0.0195%

[계산 예시] 거래금액 1,000,000원 기준
  예상 수수료(매수):      150원
  예상 수수료(매도):      150원
  증권거래세(매도):     2,000원
  총 비용(매도):        2,150원
